# Fall Detection System - V4 Pipeline

**Approach:** Pose estimation (YOLOv8-pose) + Temporal CNN classifier

**Dataset:** 420 videos from Le2i, GMDCSA24, and URFall datasets
- Train: 292 videos (70%)
- Val: 61 videos (15%)
- Test: 67 videos (15%)

**Design rationale:**
- Pose-based detection is robust to clothing, lighting, and camera angle variations
- 30-frame temporal window captures the fall dynamics, not just static poses
- 2-class classification (normal/fallen) outperformed 3-class approaches due to data imbalance
- Stratified split ensures each dataset source is represented in all splits

In [ ]:
# Install dependencies (run once)
%pip install -q ultralytics opencv-python-headless lap

In [ ]:
# Setup and imports
import os
os.environ["OPENCV_LOG_LEVEL"] = "OFF"
import sys
import json
import time
from pathlib import Path
from collections import defaultdict, deque

import cv2
import numpy as np
import torch
import torch.nn as nn
from IPython.display import display, HTML, clear_output
from ultralytics import YOLO

# Environment detection
IS_KAGGLE = os.path.exists("/kaggle/input")
IS_COLAB = "google.colab" in sys.modules

if IS_KAGGLE:
    INPUT_DIR = Path("/kaggle/input/datasets/ayushkumar10/fall-detection-v3-industrial")
    MODEL_PATH = INPUT_DIR / "fall_cnn1d_combined_best.pt"
    DATA_DIR = INPUT_DIR / "v5_fall_detection_data"  # Videos in subfolder
else:
    PROJECT_DIR = Path(".").resolve()
    EXTRAS_DIR = PROJECT_DIR / "extras"
    MODEL_PATH = EXTRAS_DIR / "outputs" / "checkpoints" / "fall_cnn1d_combined_best.pt"
    DATA_DIR = EXTRAS_DIR

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'}")
print(f"Device: {DEVICE}")
print(f"Model: {MODEL_PATH}")
print(f"Model exists: {MODEL_PATH.exists()}")

In [ ]:
# Constants
SEQ_LEN = 30
NUM_KEYPOINTS = 17
KPT_CONF_THRESH = 0.3
INPUT_FEATURES = NUM_KEYPOINTS * 3 + 7  # 17 keypoints * 3 (x,y,conf) + 7 bbox features
THRESHOLD = 0.7
CONFIRM_WINDOWS = 3


class FallCNN1D(nn.Module):
    """Temporal CNN for fall classification from pose sequences."""
    
    def __init__(self, input_features=INPUT_FEATURES, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(input_features, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.35),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.net(x))


print(f"Model architecture: CNN1D with {INPUT_FEATURES} input features")
print(f"Sequence length: {SEQ_LEN} frames")
print(f"Detection threshold: {THRESHOLD}")

In [ ]:
# Feature extraction utilities

def normalize_keypoints(keypoints, bbox_xyxy):
    """Normalize keypoints relative to bounding box."""
    x1, y1, x2, y2 = bbox_xyxy
    bw, bh = max(x2 - x1, 1.0), max(y2 - y1, 1.0)
    kpts = np.array(keypoints, dtype=np.float32).copy()
    kpts[:, 0] = (kpts[:, 0] - x1) / bw
    kpts[:, 1] = (kpts[:, 1] - y1) / bh
    return kpts


def extract_features(record):
    """Convert pose record to feature vector."""
    frame_w = float(record.get("frame_w", 640))
    frame_h = float(record.get("frame_h", 480))
    
    norm_kpts = normalize_keypoints(record["keypoints"], record["bbox_xyxy"])
    kpt_features = norm_kpts.reshape(-1).astype(np.float32)
    
    x1, y1, x2, y2 = record["bbox_xyxy"]
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    w, h = x2 - x1, y2 - y1
    
    visible_count = sum(1 for kp in record["keypoints"] if kp[2] >= KPT_CONF_THRESH)
    
    bbox_features = np.array([
        cx / frame_w,
        cy / frame_h,
        w / frame_w,
        h / frame_h,
        w / max(h, 1.0),
        (w * h) / (frame_w * frame_h),
        visible_count / NUM_KEYPOINTS,
    ], dtype=np.float32)
    
    return np.concatenate([kpt_features, bbox_features])


print("Feature extraction ready")

In [ ]:
# Fall Detection Pipeline

class FallDetector:
    """Real-time fall detection using pose estimation and temporal CNN."""
    
    def __init__(self, model_path, threshold=THRESHOLD, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.threshold = threshold
        self.seq_len = SEQ_LEN
        self.confirm_windows = CONFIRM_WINDOWS
        
        # Load CNN classifier
        ckpt = torch.load(model_path, map_location=self.device, weights_only=False)
        self.model = FallCNN1D(INPUT_FEATURES, num_classes=2).to(self.device)
        self.model.load_state_dict(ckpt["model_state_dict"])
        self.model.eval()
        
        # Load pose model
        self.pose_model = YOLO("yolov8n-pose.pt")
    
    def reset(self):
        """Reset state for new video."""
        self.buffers = defaultdict(lambda: deque(maxlen=self.seq_len))
        self.track_state = defaultdict(lambda: {
            "consecutive_fall": 0,
            "in_event": False,
            "event_start": None,
            "last_fall_frame": None,
            "max_conf": 0.0,
        })
        self.events = []
    
    @torch.no_grad()
    def predict_window(self, window):
        """Classify a sequence of poses."""
        features = np.stack([extract_features(r) for r in window])
        tensor = torch.tensor(features, dtype=torch.float32)
        tensor = tensor.transpose(0, 1).unsqueeze(0
                                                  ).to(self.device)
        probs = torch.softmax(self.model(tensor), dim=1)[0]
        fall_prob = float(probs[1].item())
        return fall_prob >= self.threshold, fall_prob
    
    def _update_event_state(self, video_name, track_id, frame_idx, is_fall, confidence):
        """Update fall event state machine."""
        state = self.track_state[track_id]
        
        if is_fall:
            state["consecutive_fall"] += 1
            state["last_fall_frame"] = frame_idx
            state["max_conf"] = max(state["max_conf"], confidence)
            
            if not state["in_event"] and state["consecutive_fall"] >= self.confirm_windows:
                state["in_event"] = True
                state["event_start"] = frame_idx
        else:
            if state["in_event"]:
                self.events.append({
                    "video_name": video_name,
                    "track_id": int(track_id),
                    "start_frame": int(state["event_start"]),
                    "end_frame": int(state["last_fall_frame"]),
                    "confidence": float(state["max_conf"]),
                })
            state["consecutive_fall"] = 0
            state["in_event"] = False
            state["event_start"] = None
            state["last_fall_frame"] = None
            state["max_conf"] = 0.0
    
    def run_on_video(self, video_path):
        """Process video file using streaming mode for proper tracking."""
        self.reset()
        video_path = str(video_path)
        video_name = Path(video_path).name
        
        # Get video dimensions
        cap = cv2.VideoCapture(video_path)
        frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        
        # Use streaming mode for proper tracking (critical!)
        results = self.pose_model.track(
            source=video_path,
            stream=True,
            persist=True,
            tracker="bytetrack.yaml",
            imgsz=640,
            conf=0.25,
            iou=0.45,
            verbose=False,
        )
        
        frame_idx = 0
        for r in results:
            frame_idx += 1
            
            if r.boxes is None or len(r.boxes) == 0:
                continue
            
            boxes = r.boxes.xyxy.cpu().numpy()
            track_ids = r.boxes.id.cpu().numpy().astype(int) if r.boxes.id is not None else np.arange(len(boxes))
            keypoints = r.keypoints.data.cpu().numpy() if r.keypoints is not None else np.zeros((len(boxes), 17, 3))
            
            for i in range(len(boxes)):
                tid = int(track_ids[i])
                kpts = keypoints[i]
                if kpts.shape[-1] == 2:
                    kpts = np.concatenate([kpts, np.ones((kpts.shape[0], 1))], axis=-1)
                
                record = {
                    "frame_idx": frame_idx,
                    "track_id": tid,
                    "bbox_xyxy": boxes[i].tolist(),
                    "keypoints": kpts.tolist(),
                    "frame_w": frame_w,
                    "frame_h": frame_h,
                }
                
                self.buffers[tid].append(record)
                
                if len(self.buffers[tid]) < self.seq_len:
                    continue
                
                is_fall, conf = self.predict_window(list(self.buffers[tid]))
                self._update_event_state(video_name, tid, frame_idx, is_fall, conf)
        
        # Finalize open events
        for tid, state in self.track_state.items():
            if state["in_event"]:
                self.events.append({
                    "video_name": video_name,
                    "track_id": int(tid),
                    "start_frame": int(state["event_start"]),
                    "end_frame": int(state["last_fall_frame"]),
                    "confidence": float(state["max_conf"]),
                })
        
        return self.events
    
    def process_frame_webcam(self, frame):
        """Process single frame for webcam mode."""
        if not hasattr(self, '_webcam_frame_idx'):
            self._webcam_frame_idx = 0
            self._webcam_buffers = defaultdict(lambda: deque(maxlen=self.seq_len))
            self._webcam_state = defaultdict(lambda: {"consecutive_fall": 0, "in_event": False})
        
        self._webcam_frame_idx += 1
        frame_h, frame_w = frame.shape[:2]
        
        results = self.pose_model.track(
            source=frame,
            persist=True,
            tracker="bytetrack.yaml",
            verbose=False,
        )
        
        alerts = []
        for r in results:
            if r.boxes is None or len(r.boxes) == 0:
                continue
            
            boxes = r.boxes.xyxy.cpu().numpy()
            track_ids = r.boxes.id.cpu().numpy().astype(int) if r.boxes.id is not None else np.arange(len(boxes))
            keypoints = r.keypoints.data.cpu().numpy() if r.keypoints is not None else np.zeros((len(boxes), 17, 3))
            
            for i, (box, tid, kpts) in enumerate(zip(boxes, track_ids, keypoints)):
                record = {
                    "frame_idx": self._webcam_frame_idx,
                    "track_id": int(tid),
                    "bbox_xyxy": box.tolist(),
                    "keypoints": kpts.tolist(),
                    "frame_w": frame_w,
                    "frame_h": frame_h,
                }
                
                self._webcam_buffers[tid].append(record)
                
                if len(self._webcam_buffers[tid]) >= self.seq_len:
                    is_fall, conf = self.predict_window(list(self._webcam_buffers[tid]))
                    state = self._webcam_state[tid]
                    
                    if is_fall:
                        state["consecutive_fall"] += 1
                        if state["consecutive_fall"] >= self.confirm_windows:
                            alerts.append({"track_id": tid, "bbox": box, "confidence": conf})
                    else:
                        state["consecutive_fall"] = 0
        
        return alerts, results


# Initialize detector
detector = FallDetector(MODEL_PATH, threshold=THRESHOLD, device=DEVICE)
print(f"Fall detector initialized (threshold={THRESHOLD})")

## Training

Train the FallCNN1D model on the training set with validation monitoring.

In [ ]:
# Training data extraction and dataset preparation

import torch.utils.data as data
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
import pickle
import gc

skipped_stats = {"not_found": 0, "no_timestamps": 0, "no_tracks": 0, "gmc_error": 0}

def resolve_video_path(video_path):
    """Find video file across different directory structures."""
    p = Path(video_path)
    if p.exists():
        return p
    
    if IS_KAGGLE:
        kaggle_path = DATA_DIR / video_path
        if kaggle_path.exists():
            return kaggle_path
    
    filename = p.name
    search_dirs = [DATA_DIR] if IS_KAGGLE else [
        EXTRAS_DIR / "archive",
        EXTRAS_DIR / "GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos-master",
        EXTRAS_DIR / "outputs" / "urfall_mp4",
    ]
    
    for dir_path in search_dirs:
        if dir_path.exists():
            matches = list(dir_path.rglob(filename))
            if matches:
                return matches[0]
    return None


def extract_training_data(manifest_path, pose_model, data_dir=None, checkpoint_path=None):
    """Extract pose features from all videos in manifest with checkpointing."""
    global skipped_stats
    skipped_stats = {"not_found": 0, "no_timestamps": 0, "no_tracks": 0, "gmc_error": 0}
    
    with open(manifest_path) as f:
        manifest = json.load(f)
    
    # Load checkpoint if exists
    processed_uids = set()
    all_windows = []
    all_labels = []
    
    if checkpoint_path and Path(checkpoint_path).exists():
        with open(checkpoint_path, 'rb') as f:
            ckpt = pickle.load(f)
            all_windows = ckpt['windows']
            all_labels = ckpt['labels']
            processed_uids = ckpt['processed_uids']
            print(f"Resumed from checkpoint: {len(processed_uids)} videos, {len(all_labels)} windows")
    
    save_every = 20  # Save checkpoint every N videos
    videos_since_save = 0
    
    for video_info in tqdm(manifest["videos"], desc="Extracting features"):
        video_uid = video_info["video_uid"]
        
        if video_uid in processed_uids:
            continue
            
        video_path = resolve_video_path(video_info["video_path"])
        if video_path is None:
            skipped_stats["not_found"] += 1
            processed_uids.add(video_uid)
            continue
        
        has_fall = video_info.get("has_fall", False)
        fall_start = video_info.get("fall_start_frame")
        fall_end = video_info.get("fall_end_frame")
        
        if has_fall and (fall_start is None or fall_end is None or fall_start <= 0 or fall_end <= 0):
            skipped_stats["no_timestamps"] += 1
            processed_uids.add(video_uid)
            continue
        
        try:
            cap = cv2.VideoCapture(str(video_path))
            frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            cap.release()
            
            buffers = defaultdict(list)
            results = pose_model.track(
                source=str(video_path),
                stream=True,
                persist=True,
                tracker="bytetrack.yaml",  # ByteTrack avoids GMC crashes
                imgsz=640,
                conf=0.25,
                verbose=False,
            )
            
            frame_idx = 0
            for r in results:
                frame_idx += 1
                if r.boxes is None or len(r.boxes) == 0:
                    continue
                
                boxes = r.boxes.xyxy.cpu().numpy()
                track_ids = r.boxes.id.cpu().numpy().astype(int) if r.boxes.id is not None else np.arange(len(boxes))
                keypoints = r.keypoints.data.cpu().numpy() if r.keypoints is not None else np.zeros((len(boxes), 17, 3))
                
                for i in range(len(boxes)):
                    tid = int(track_ids[i])
                    kpts = keypoints[i]
                    if kpts.shape[-1] == 2:
                        kpts = np.concatenate([kpts, np.ones((kpts.shape[0], 1))], axis=-1)
                    
                    record = {
                        "frame_idx": frame_idx,
                        "track_id": tid,
                        "bbox_xyxy": boxes[i].tolist(),
                        "keypoints": kpts.tolist(),
                        "frame_w": frame_w,
                        "frame_h": frame_h,
                    }
                    buffers[tid].append(record)
            
            if not buffers:
                skipped_stats["no_tracks"] += 1
                processed_uids.add(video_uid)
                continue
            
            for tid, records in buffers.items():
                if len(records) < SEQ_LEN:
                    continue
                
                for start in range(0, len(records) - SEQ_LEN + 1, SEQ_LEN // 2):
                    window = records[start:start + SEQ_LEN]
                    features = np.stack([extract_features(r) for r in window])
                    
                    window_start = window[0]["frame_idx"]
                    window_end = window[-1]["frame_idx"]
                    
                    if has_fall:
                        overlap = window_start <= fall_end and window_end >= fall_start
                        label = 1 if overlap else 0
                    else:
                        label = 0
                    
                    all_windows.append(features)
                    all_labels.append(label)
            
            processed_uids.add(video_uid)
            videos_since_save += 1
            
            # Periodic checkpoint save
            if checkpoint_path and videos_since_save >= save_every:
                with open(checkpoint_path, 'wb') as f:
                    pickle.dump({
                        'windows': all_windows,
                        'labels': all_labels,
                        'processed_uids': processed_uids
                    }, f)
                videos_since_save = 0
                gc.collect()
                
        except Exception as e:
            if "GMC" in str(e) or "orb" in str(e).lower():
                skipped_stats["gmc_error"] += 1
            else:
                print(f"Error processing {video_uid}: {e}")
            processed_uids.add(video_uid)
            gc.collect()
            continue
    
    # Final save
    if checkpoint_path:
        with open(checkpoint_path, 'wb') as f:
            pickle.dump({
                'windows': all_windows,
                'labels': all_labels,
                'processed_uids': processed_uids
            }, f)
    
    print(f"\nSkipped: {skipped_stats['not_found']} not found, {skipped_stats['no_timestamps']} no timestamps, {skipped_stats['no_tracks']} no tracks, {skipped_stats['gmc_error']} GMC errors")
    return np.array(all_windows), np.array(all_labels)


class FallDataset(data.Dataset):
    def __init__(self, windows, labels):
        self.windows = torch.tensor(windows, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.windows[idx].T, self.labels[idx]


print("Training utilities defined")

In [ ]:
# Run training

# Paths for train/val/test manifests
if IS_KAGGLE:
    TRAIN_MANIFEST = INPUT_DIR / "train_manifest_kaggle.json"
    VAL_MANIFEST = INPUT_DIR / "val_manifest_kaggle.json"
    TEST_MANIFEST = INPUT_DIR / "test_manifest_kaggle.json"
    TRAIN_CKPT = Path("/kaggle/working/train_features.pkl")
    VAL_CKPT = Path("/kaggle/working/val_features.pkl")
else:
    TRAIN_MANIFEST = PROJECT_DIR / "train_manifest.json"
    VAL_MANIFEST = PROJECT_DIR / "val_manifest.json"
    TEST_MANIFEST = PROJECT_DIR / "test_manifest.json"
    TRAIN_CKPT = PROJECT_DIR / "train_features.pkl"
    VAL_CKPT = PROJECT_DIR / "val_features.pkl"

# Training hyperparameters
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

# Check if we should train
if TRAIN_MANIFEST.exists():
    print("Loading pose model for feature extraction...")
    pose_model = YOLO("yolov8n-pose.pt")
    
    # Extract training data (with checkpointing)
    print(f"\nExtracting training data from {TRAIN_MANIFEST}...")
    train_windows, train_labels = extract_training_data(TRAIN_MANIFEST, pose_model, checkpoint_path=TRAIN_CKPT)
    print(f"Training: {len(train_labels)} windows ({sum(train_labels)} falls, {len(train_labels) - sum(train_labels)} normal)")
    
    # Extract validation data (with checkpointing)
    print(f"\nExtracting validation data from {VAL_MANIFEST}...")
    val_windows, val_labels = extract_training_data(VAL_MANIFEST, pose_model, checkpoint_path=VAL_CKPT)
    print(f"Validation: {len(val_labels)} windows ({sum(val_labels)} falls, {len(val_labels) - sum(val_labels)} normal)")
    
    # Create datasets and loaders
    train_dataset = FallDataset(train_windows, train_labels)
    val_dataset = FallDataset(val_windows, val_labels)
    
    # Class weights for imbalanced data
    n_normal = (train_labels == 0).sum()
    n_fall = (train_labels == 1).sum()
    weight_normal = len(train_labels) / (2 * n_normal) if n_normal > 0 else 1.0
    weight_fall = len(train_labels) / (2 * n_fall) if n_fall > 0 else 1.0
    class_weights = torch.tensor([weight_normal, weight_fall], dtype=torch.float32).to(DEVICE)
    print(f"\nClass weights: normal={weight_normal:.2f}, fall={weight_fall:.2f}")
    
    train_loader = data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = data.DataLoader(val_dataset, batch_size=BATCH_SIZE)
    
    # Initialize model, optimizer, scheduler
    model = FallCNN1D(INPUT_FEATURES, num_classes=2).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    
    best_val_recall = 0.0
    best_state = None
    history = []
    
    print(f"\nTraining for {EPOCHS} epochs...")
    for epoch in range(EPOCHS):
        # Train
        model.train()
        train_loss = 0.0
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(y)
        train_loss /= len(train_dataset)
        
        # Validate
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for X, y in val_loader:
                X = X.to(DEVICE)
                logits = model(X)
                preds = logits.argmax(dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(y.numpy())
        
        val_preds = np.array(val_preds)
        val_true = np.array(val_true)
        
        tp = ((val_preds == 1) & (val_true == 1)).sum()
        fn = ((val_preds == 0) & (val_true == 1)).sum()
        fp = ((val_preds == 1) & (val_true == 0)).sum()
        
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        scheduler.step()
        
        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_recall": recall,
            "val_precision": precision,
        })
        
        # Save best
        if recall > best_val_recall:
            best_val_recall = recall
            best_state = model.state_dict().copy()
            marker = " *"
        else:
            marker = ""
        
        print(f"Epoch {epoch+1:02d}: loss={train_loss:.4f} | val_recall={recall:.2%} | val_precision={precision:.2%}{marker}")
    
    # Save best model
    if best_state is not None:
        model.load_state_dict(best_state)
        save_path = Path("/kaggle/working/fall_cnn1d_v4_best.pt") if IS_KAGGLE else (EXTRAS_DIR / "outputs" / "checkpoints" / "fall_cnn1d_v4_best.pt")
        save_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({
            "model_state_dict": best_state,
            "threshold": THRESHOLD,
            "seq_len": SEQ_LEN,
            "input_features": INPUT_FEATURES,
        }, save_path)
        print(f"\nBest model saved to {save_path} (val_recall={best_val_recall:.2%})")
        
        # Save training history
        history_path = Path("/kaggle/working/training_history.json") if IS_KAGGLE else (PROJECT_DIR / "training_history.json")
        with open(history_path, 'w') as f:
            json.dump(history, f, indent=2)
        print(f"Training history saved to {history_path}")
        
        MODEL_PATH = save_path
else:
    print(f"Train manifest not found at {TRAIN_MANIFEST}")
    print("Skipping training - using existing model")

## Validation on Test Dataset

In [ ]:
# Load test manifest for final evaluation

if TEST_MANIFEST.exists():
    test_manifest = json.load(open(TEST_MANIFEST))
    total_falls = sum(1 for v in test_manifest['videos'] if v.get('has_fall'))
    total_normal = len(test_manifest['videos']) - total_falls
    print(f"Loaded test manifest: {len(test_manifest['videos'])} videos ({total_falls} falls, {total_normal} normal)")
else:
    test_manifest = None
    print("Test manifest not found - skipping final evaluation")

In [ ]:
# Run final evaluation on test set

if test_manifest:
    # Re-initialize detector with trained model
    detector = FallDetector(MODEL_PATH, threshold=THRESHOLD, device=DEVICE)
    print(f"Evaluating with model: {MODEL_PATH}")
    
    results = []
    
    for i, video_info in enumerate(test_manifest["videos"]):
        video_path = resolve_video_path(video_info["video_path"])
        if video_path is None:
            continue
        
        has_fall_gt = video_info.get("has_fall", False)
        fall_start_gt = video_info.get("fall_start_frame")
        fall_end_gt = video_info.get("fall_end_frame")
        
        events = detector.run_on_video(video_path)
        detected = len(events) > 0
        
        # Calculate detection delay if applicable
        delay = None
        if has_fall_gt and detected and fall_start_gt:
            first_detection = min(e["start_frame"] for e in events)
            delay = first_detection - fall_start_gt
        
        result = {
            "video": video_info["video_uid"],
            "has_fall_gt": has_fall_gt,
            "detected": detected,
            "tp": has_fall_gt and detected,
            "fn": has_fall_gt and not detected,
            "fp": not has_fall_gt and detected,
            "delay": delay,
        }
        results.append(result)
        
        # Progress
        status = "TP" if result["tp"] else "FN" if result["fn"] else "FP" if result["fp"] else "TN"
        delay_str = f" (delay={delay})" if delay is not None else ""
        print(f"[{i+1}/{len(test_manifest['videos'])}] {video_info['video_uid'][:40]}... {status}{delay_str}")
    
    # Summary
    tp = sum(r["tp"] for r in results)
    fn = sum(r["fn"] for r in results)
    fp = sum(r["fp"] for r in results)
    tn = len(results) - tp - fn - fp
    
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    delays = [r["delay"] for r in results if r["delay"] is not None]
    avg_delay = np.mean(delays) if delays else 0
    
    print("\n" + "="*60)
    print("TEST SET RESULTS (held out)")
    print("="*60)
    print(f"Total videos: {len(results)}")
    print(f"True Positives: {tp}")
    print(f"False Negatives: {fn}")
    print(f"False Positives: {fp}")
    print(f"True Negatives: {tn}")
    print(f"\nRecall: {recall*100:.1f}%")
    print(f"Precision: {precision*100:.1f}%")
    print(f"F1 Score: {f1*100:.1f}%")
    print(f"False Alarms: {fp}")
    print(f"Avg Detection Delay: {avg_delay:.1f} frames")
    print("="*60)
else:
    print("Skipping test evaluation - no manifest")

## Live Inference - Webcam or Video Upload

In [ ]:
# Inference utilities

def draw_detections(frame, alerts, pose_results=None):
    """Draw bounding boxes and fall alerts on frame."""
    annotated = frame.copy()
    
    # Draw pose skeletons if available
    if pose_results and len(pose_results) > 0:
        annotated = pose_results[0].plot()
    
    # Draw fall alerts
    for alert in alerts:
        x1, y1, x2, y2 = map(int, alert["bbox"])
        cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 0, 255), 3)
        label = f"FALL DETECTED ({alert['confidence']:.2f})"
        cv2.putText(annotated, label, (x1, y1-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    
    return annotated


def run_on_webcam(detector, camera_id=0, max_frames=None):
    """Run fall detection on webcam feed (uses frame-by-frame mode)."""
    cap = cv2.VideoCapture(camera_id)
    
    if not cap.isOpened():
        print("Could not open webcam")
        return []
    
    print("Press 'q' to quit")
    frame_count = 0
    fps_start = time.time()
    all_alerts = []
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            alerts, pose_results = detector.process_frame_webcam(frame)
            annotated = draw_detections(frame, alerts, pose_results)
            all_alerts.extend(alerts)
            
            # FPS counter
            frame_count += 1
            elapsed = time.time() - fps_start
            fps = frame_count / elapsed if elapsed > 0 else 0
            cv2.putText(annotated, f"FPS: {fps:.1f}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            
            cv2.imshow("Fall Detection", annotated)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
            
            if max_frames and frame_count >= max_frames:
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()
    
    print(f"Processed {frame_count} frames at {fps:.1f} FPS")
    print(f"Fall alerts: {len(all_alerts)}")
    return all_alerts


def run_on_video_file_visualize(detector, video_path, output_path=None):
    """Process video with visualization (slower, for demos)."""
    cap = cv2.VideoCapture(str(video_path))
    
    if not cap.isOpened():
        print(f"Could not open video: {video_path}")
        return None
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    
    # First, run detection (uses streaming mode for accuracy)
    print(f"Processing {Path(video_path).name}...")
    events = detector.run_on_video(video_path)
    print(f"Found {len(events)} fall events")
    
    # If output requested, create annotated video
    if output_path:
        print(f"Creating annotated video...")
        cap = cv2.VideoCapture(str(video_path))
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))
        
        # Build frame ranges with active falls
        fall_frames = set()
        for event in events:
            for f in range(event['start_frame'], event['end_frame'] + 1):
                fall_frames.add(f)
        
        frame_idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1
            
            if frame_idx in fall_frames:
                # Add FALL warning
                cv2.putText(frame, "FALL DETECTED", (50, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)
                cv2.rectangle(frame, (40, 20), (350, 70), (0, 0, 255), 2)
            
            writer.write(frame)
        
        cap.release()
        writer.release()
        print(f"Saved to: {output_path}")
    
    return events


print("Inference functions ready:")
print("  - detector.run_on_video(path): Fast evaluation (streaming mode)")
print("  - run_on_webcam(detector): Live webcam detection")
print("  - run_on_video_file_visualize(detector, path, output): Create annotated video")

In [ ]:
# Upload and process video (Kaggle/Colab)

if IS_KAGGLE or IS_COLAB:
    print("Upload a video file to test:")
    
    if IS_COLAB:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            video_file = list(uploaded.keys())[0]
            events = run_on_video_file(detector, video_file, output_path="output_annotated.mp4")
    else:
        # Kaggle - use file from input
        print("On Kaggle, specify video path manually:")
        print("  events = run_on_video_file(detector, '/path/to/video.mp4')")
else:
    print("Local environment - options:")
    print("  1. Webcam: run_on_webcam(detector)")
    print("  2. Video:  run_on_video_file(detector, 'path/to/video.mp4')")

In [ ]:
# Example: Process a specific video
# Uncomment and modify as needed

# video_path = "path/to/your/video.mp4"
# events = run_on_video_file(detector, video_path, output_path="output.mp4", show=False)

In [ ]:
# Webcam demo (local only)
# Uncomment to run

# if not IS_KAGGLE and not IS_COLAB:
#     events = run_on_webcam(detector)

## Summary

**System Performance:**
- Recall: ~91% (catches 9 out of 10 falls)
- Precision: ~67% (1 in 3 alerts may be false)
- Throughput: ~30 FPS on GPU
- Detection latency: 1-2 seconds after fall

**Technical approach:**
1. YOLOv8-pose extracts 17 body keypoints per person per frame
2. ByteTrack maintains person identity across frames
3. CNN1D classifies 30-frame pose sequences as normal/fallen
4. 3-window confirmation reduces false triggers

**Production considerations:**
- Current precision (67%) is acceptable for POC but needs improvement for deployment
- More diverse training data would improve generalization
- Site-specific fine-tuning recommended for best results